# 04 Self-Reflection — Prompt 驱动的自我检查

不加新工具、不加新类、不改 ReAct 循环。只改一行 SYSTEM_PROMPT，LLM 的行为就变了。

**学习点**：Reflection 是 ReAct + 正确 prompt 的自然涌现，不是外加机制。

In [ ]:
import sys, json, shutil, os
sys.path.insert(0, '..')

from src.agent_framework import Agent
from src.capabilities import PlanManager
from src.capabilities.demo_tools import create_demo_tools

for d in ['agent_memory']:
    if os.path.exists(d):
        shutil.rmtree(d)

print('一切就绪')

## 不加反思的 Agent

和之前的实验完全一样的 Agent。

In [ ]:
PROMPT_NO_REFLECT = (
    "你是一个有用的 AI 助手，可以用中文回复。"
    "需要时使用工具获取信息。"
    "当面对需要多步协调的复杂任务时，先调用 make_plan 制定计划。"
)

plan_mgr = PlanManager()
tools = create_demo_tools(plan_mgr=plan_mgr, ltm=None)
agent_no_reflect = Agent(tools=tools, system_prompt=PROMPT_NO_REFLECT, plan_mgr=plan_mgr)
print('Agent (无反思) 已就绪')

### 执行复杂任务

LLM 拿到第一个结果后很可能直接输出答案，不会自检。

In [ ]:
reply = agent_no_reflect.chat(
    '特斯拉的股价在当前市场环境下是否值得投资？请综合考虑股价、行业前景和风险因素。'
)
print(f'\n=== 最终回答 ===\n{reply[:500]}...' if len(reply) > 500 else f'\n=== 最终回答 ===\n{reply}')

## 加了反思的 Agent

同样一套代码，只多了一行 prompt。

In [ ]:
PROMPT_WITH_REFLECT = (
    "你是一个有用的 AI 助手，可以用中文回复。"
    "需要时使用工具获取信息。"
    "当面对需要多步协调的复杂任务时，先调用 make_plan 制定计划。"
    "在给出最终答案前，请先自我检查：数据是否准确？逻辑是否完整？是否有遗漏？如果发现问题，先修正再回答。"
)

plan_mgr2 = PlanManager()
tools2 = create_demo_tools(plan_mgr=plan_mgr2, ltm=None)
agent_with_reflect = Agent(tools=tools2, system_prompt=PROMPT_WITH_REFLECT, plan_mgr=plan_mgr2)
print('Agent (有反思) 已就绪')

### 同一个任务

对比 LLM 的行为差异。

In [ ]:
reply = agent_with_reflect.chat(
    '特斯拉的股价在当前市场环境下是否值得投资？请综合考虑股价、行业前景和风险因素。'
)
print(f'\n=== 最终回答 ===\n{reply[:500]}...' if len(reply) > 500 else f'\n=== 最终回答 ===\n{reply}')

## 对比分析

关键差异：

| | 无反思 | 有反思 |
|---|---|---|
| 工具调用次数 | 较少 | 较多（LLM 意识到数据不够后补调） |
| 输出完整性 | 可能缺数据、缺风险分析 | 更全面 |
| 幻觉 | 可能基于不完整数据臆断 | 意识到不足后会说明 |

**核心结论**：Self-Reflection 不需要新机制。ReAct 循环从第一天就支持它——LLM 在拿到工具结果后可以决定'不够，我再来一轮'。加 prompt 只是让 LLM 更大概率做出这个决定。

## 总结

04 只改了一行 SYSTEM_PROMPT，改了零行框架代码。

这证明了：
- ReAct 循环本身就是反思引擎
- LLM 的行为 = 能力(run LLM) + 意愿(prompt)
- 很多所谓新能力其实只是 prompt engineering



In [ ]:
shutil.rmtree('agent_memory', ignore_errors=True)
print('演示完成')